# 08 — Pseudobulk 差异表达分析 (DESeq2)

## 核心理念

单细胞级别的差异表达分析存在 **pseudoreplication 问题**：
数万个细胞中即使 fold change 很小，p-value 也会因为样本量大得惊人——但这不反映
真实的生物学效应，而是把同一个样本内部的细胞间随机波动误当成了独立观测。
**解决方法：在 sample 层面做 pseudobulk 聚合**，每个 (sample x cell-type) 组合
合并为一个"伪样本"，然后用 DESeq2 做严格的统计推断。

## 工作流

1. Python (`decoupler.get_pseudobulk`): 把单细胞矩阵聚合为 pseudobulk counts
2. 导出 counts.csv + metadata.csv → 工作目录
3. `subprocess` 调用 `Rscript scripts/deseq2_contrast.R` 执行 DESeq2
4. 读回 DESeq2 结果表 → 写入 adata.uns + 导出 CSV

## 重要声明 (disclaimer)

> 本 notebook 做的是 **sample (不是 per-cell) 层面的 DEG**。
> Pseudobulk 聚合虽然稀释了单细胞的极端噪声，但它并不消除数据集间的系统性偏差
> ——不同数据集用不同 protocol 测序、不同时间点采集、不同医院处理，
> 在 pseudobulk 层面这些 batch effect 仍然存在。
> 数据干净、batch 不严重时，pseudobulk DESeq2 是目前公认最稳健的
> 跨条件 DEG 方法之一（Squair et al., 2021; Crowell et al., 2020）。
> 但如果你的 batch 结构与 disease 组完全混在一起的（confounded），
> DESeq2 也无法区分 batch 效应和真实的疾病效应——这是一个实验设计问题，
> 不是分析方法问题。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：03（标准化，counts layer）+ 06（注释），读 `06_annotated_v*.h5ad`
- **下游**：下游分析（火山图 / 跨条件对比），产出 `08_pseudobulk_deg_v*.h5ad` + DESeq2 结果 CSV

### 为什么要迭代回跑？
Pseudobulk 差异表达分析 (DESeq2) 的结果是下游分析和 PI 生物学判断的基础。如果在后续分析中发现：
- DEG 阈值过高导致遗漏关键基因、过低导致假阳性
- 通路富集缺少预期应出现的生物学通路
- 调控网络缺少已知的主控转录因子
- CNV 信号不符合病理学预期
可能需要调整本 stage 的参数重新计算。

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`（旧版不覆盖）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改以下参数后重跑本 notebook）：
- `CELL_TYPE_COL` — 切换细胞类型分组列
- `DISEASE_CONTRASTS` — 调整疾病对比对
- `PSEUDOBULK_MIN_CELLS` / `PSEUDOBULK_MIN_SAMPLES` — 过滤阈值
- DESeq2 R 脚本参数（`scripts/deseq2_contrast.R` 内部）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为结果可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：下游 notebook 的 `UPSTREAM_PATH` 指向你决定采用的版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"08_pseudobulk_deg"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询「Pseudobulk 差异表达分析 (DESeq2) 有哪些版本？哪些依赖 06_annotated_v1？」，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH              — 06 注释结果 h5ad
# OUTPUT_PATH         — 本 stage 产出 checkpoint 路径。
#                        版本号 _v1 与 adata.uns['version'] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# SAMPLE_COL                 — 样本 ID obs 列
# CELL_TYPE_COL              — 细胞类型 obs 列（用于 groups_col）
#                              Nancang fixture: cell_type_final_v1 为空（PI 未标注），
#                              使用 leiden_res_0.6 作为替代分组。
# DISEASE_COL                — 疾病/条件 obs 列（用于 DESeq2 分组因子）
#                              **重要提醒**：此列需在 01/02 已填充。
#                              多数真实数据集的 obs 中不包含此列；若列为空或不存在，
#                              请先在 02 的 obs 中填好疾病分期列并修改本 PARAMS。
#                              常见实际列名：disease_stage、condition、group 等。
# DISEASE_CONTRASTS          — DESeq2 对比列表 [(numerator, denominator), ...]
# PSEUDOBULK_MIN_CELLS       — 每个 (sample, cell_type) 组合最少细胞数
# PSEUDOBULK_MIN_SAMPLES     — 每个 cell_type 最少样本数
# DESEQ2_RSCRIPT             — DESeq2 R 脚本路径
# DESEQ2_WORKDIR             — 临时文件工作目录

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
OUTPUT_PATH   = "results/08_pseudobulk_deg_v1.h5ad"

SAMPLE_COL    = "sample_id"
CELL_TYPE_COL = "leiden_res_0.6"      # Nancang fixture: 用 Leiden 簇替代 cell_type

DISEASE_COL   = "disease"  # best-effort 默认值——多数数据集可能为空，PI 必须核对实际 obs 列名

DISEASE_CONTRASTS = [
    ("CAG", "normal"),
    ("IM", "CAG"),
    ("dysplasia", "IM"),
]

PSEUDOBULK_MIN_CELLS   = 10
PSEUDOBULK_MIN_SAMPLES = 3

DESEQ2_RSCRIPT  = "scripts/deseq2_contrast.R"
DESEQ2_WORKDIR  = "results/_deseq2_tmp"  # 临时文件，不入 git

In [ ]:
# === setup：sys.path + 导入 + 加载上游 ===

# 1. 项目根目录定位 + sys.path
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs(DESEQ2_WORKDIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")
# 2. 导入依赖
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import shutil
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

# R 环境守卫——Rscript 不可用时后续 DESeq2 cell 优雅跳过。
# rscript_bin() 找不到 Rscript 时会抛 RuntimeError，
# 用 try-except 优雅降级：设 _R_AVAILABLE=False 让后续 DESeq2 步骤跳过。
from scrna_integration.platform import check_r_available
_RSCRIPT_PATH, _R_AVAILABLE = check_r_available()

# 检查 DESeq2.R 脚本是否存在
if _R_AVAILABLE and not os.path.exists(DESEQ2_RSCRIPT):
    print(f"WARNING: DESeq2 R 脚本不存在: {DESEQ2_RSCRIPT}")
    _R_AVAILABLE = False

print(f"scanpy {sc.__version__}")
# 3. 加载上游 06 输出
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")

# 验证必需列（DISEASE_COL 为可选项——多数数据集没有疾病信息）
_required_cols = [SAMPLE_COL, CELL_TYPE_COL]
_missing = [c for c in _required_cols if c not in adata.obs.columns]
if _missing:
    print(f"WARNING: 缺少 obs 列: {_missing}")
    print("请检查 PARAMS 中的列名是否与 06 输出匹配")
else:
    print(f"样本数: {adata.obs[SAMPLE_COL].nunique()}")
    print(f"细胞类型（{CELL_TYPE_COL}）: {adata.obs[CELL_TYPE_COL].nunique()}")
    if DISEASE_COL in adata.obs.columns:
        print(f"疾病组: {sorted(adata.obs[DISEASE_COL].dropna().unique())}")
    else:
        print(f"DISEASE_COL '{DISEASE_COL}' 不在 obs 中——DESeq2 对比将跳过")

## Pseudobulk 聚合

用 `decoupler.get_pseudobulk` 把单细胞 counts 矩阵聚合到 sample 层面。
每个 **(sample_id, cell_type)** 组合的所有细胞合并为一个"伪样本"，
其 counts = 该组内所有细胞的原始 UMI counts 之和。

**为什么用 `layer="counts"`（原始整数 counts）而不是 `layer=None`（log-normalized）？**
这是 DESeq2 的硬性要求。DESeq2 内部用 negative binomial 分布建模计数数据，
该分布的两个参数——均值和 dispersion——都定义在**非负整数**域上。
log-normalized 数据是经过 log(1+x) 变换的连续浮点数（范围 0~10），
直接喂给 DESeq2 会破坏 NB 分布的数学假设，导致 p 值/logFC 全部不可靠。
**只有在 `layer="counts"` 取得原始 UMI counts**，DESeq2 才有正确的输入
进行严格的样本层面统计推断。

**上游契约（03_normalized.ipynb）**：原始整数 counts 保存于
`adata.layers["counts"]`——这是 03 在 `normalize_total` 之前
从 `adata.X` 做的 CSR 拷贝。若 03 未跑或契约变更，本 cell 会因
layer key 不存在而报错，而非静默使用错误数据。

**为什么用 sum 而不是 mean？** DESeq2 需要 count-level 的整数输入来做
negative binomial 建模；mean 会把计数变成小数，破坏分布假设。
**为什么 decoupler？** 它封装了聚合逻辑（groupby-sum + 稀疏矩阵高效求和），
比自己手写 groupby+sum 更稳健且内存友好。

In [ ]:
# Pseudobulk 聚合——sample 层面。
# dc.pp.pseudobulk 返回一个 AnnData，其中 .X 是 sample×gene 的 counts 矩阵，
# .obs 包含每个 pseudobulk 样本的元数据（sample_id, cell_type, disease 等）。
# 关键：layer="counts" 取 adata.layers["counts"]（03 保留的原始整数 UMI counts）。
# DESeq2 的 negative binomial 模型要求整数输入；log-normalized 的 adata.X
# 是连续浮点数，会破坏 NB 分布假设，导致 p 值/logFC 不可靠。
# decoupler 2.1.6+ API：dc.pp.pseudobulk(adata, sample_col, groups_col, layer, mode='sum')
import decoupler as dc

if all(c in adata.obs.columns for c in _required_cols):
    print("Pseudobulk 聚合中...")
    print(f"  sample_col={SAMPLE_COL}, groups_col={CELL_TYPE_COL}")

    pdata = dc.pp.pseudobulk(
        adata,
        sample_col=SAMPLE_COL,
        groups_col=CELL_TYPE_COL,   # 传列名字符串（decoupler 2.1.6 要求 str）
        layer="counts",             # 原始整数 UMI counts（03 保留在 adata.layers["counts"]）
        mode="sum",
    )
    # 后置过滤：移除细胞数不足的组合（decoupler 2.1.6 不再支持 min_cells/min_samples 参数，
    # 改为使用 psbulk QC 指标后置过滤）
    if "psbulk_n_cells" in pdata.obs.columns:
        _mask_cells = pdata.obs["psbulk_n_cells"] >= PSEUDOBULK_MIN_CELLS
        pdata = pdata[_mask_cells].copy()
    if SAMPLE_COL in pdata.obs.columns and PSEUDOBULK_MIN_SAMPLES > 1:
        _sample_counts = pdata.obs.groupby(CELL_TYPE_COL)[SAMPLE_COL].nunique()
        _keep_ct = _sample_counts[_sample_counts >= PSEUDOBULK_MIN_SAMPLES].index
        pdata = pdata[pdata.obs[CELL_TYPE_COL].isin(_keep_ct)].copy()

    print(f"Pseudobulk 完成: {pdata.n_obs} pseudobulk samples x {pdata.n_vars} genes")
    if CELL_TYPE_COL in pdata.obs.columns:
        print(f"  细胞类型: {pdata.obs[CELL_TYPE_COL].nunique()}")
    if SAMPLE_COL in pdata.obs.columns:
        print(f"  样本: {pdata.obs[SAMPLE_COL].nunique()}")
    if DISEASE_COL in pdata.obs.columns:
        print(f"  疾病组分布:")
        print(pdata.obs[DISEASE_COL].value_counts().to_string())
    else:
        print(f"  DISEASE_COL '{DISEASE_COL}' 不在 pseudobulk obs 中——DESeq2 对比将跳过")
else:
    print("缺少必需列，跳过 pseudobulk 聚合")
    pdata = None

In [ ]:
# 导出 counts + metadata 为 CSV，供 Rscript 读取。
if pdata is not None and pdata.n_obs > 0:
    # counts: genes × samples 矩阵（DESeq2 期望行=基因、列=样本）
    _counts_df = pd.DataFrame(
        pdata.X.T if sp.issparse(pdata.X) else pdata.X.T,
        index=pdata.var_names,
        columns=pdata.obs_names,
    )
    _counts_csv = os.path.join(DESEQ2_WORKDIR, "counts.csv")
    _counts_df.to_csv(_counts_csv)
    print(f"Counts 已导出: {_counts_csv} ({_counts_df.shape[0]} genes x {_counts_df.shape[1]} samples)")

    # metadata：样本 × 协变量
    _meta_csv = os.path.join(DESEQ2_WORKDIR, "metadata.csv")
    pdata.obs.to_csv(_meta_csv)
    print(f"Metadata 已导出: {_meta_csv}")
else:
    print("pdata 为空或 None，跳过导出")

## DESeq2 差异表达分析（subprocess 调用 Rscript）

调用 `scripts/deseq2_contrast.R`，对每个疾病对比执行 DESeq2 差异表达分析。

**为什么用 DESeq2？** DESeq2 是 bulk RNA-seq 差异表达分析的公认金标准方法
（Love et al., 2014）。它的核心优势在于用 negative binomial 分布建模 count
数据，天然处理 RNA-seq 数据中的过度离散（overdispersion）问题——基因表达的
方差通常远大于均值，简单的 t 检验或线性模型会产生大量假阳性。DESeq2 通过
empirical Bayes shrinkage 同时估计每个基因的 dispersion，在小样本量场景下
尤其稳健。

**为什么用 subprocess 而非 rpy2 直接调 R？** rpy2 桥接 Python 与 R 虽然方便，
但在处理大型 AnnData 对象时容易因 R 与 Python 依赖版本升级导致崩溃，且错误栈
同时涉及两种语言，极难排查。subprocess 调用独立 Rscript 将 Python 与 R 进程
完全隔离——R 脚本可独立调试、独立测试，出问题时只需排查 R 一侧，工艺上更稳健。

**输入/输出**：
- 输入：`results/_deseq2_tmp/counts.csv`（pseudobulk 整数 count 矩阵）+
  `metadata.csv`（样本表型信息）
- 输出：`results/_deseq2_tmp/deg_{numerator}_vs_{denominator}.csv`（DESeq2 完整结果表，
  含 `log2FoldChange`、`pvalue`、`padj` 等列）


In [ ]:
# DESeq2 subprocess——逐对比调用 Rscript。
# 每个对比产出：results/_deseq2_tmp/deg_{num}_vs_{den}.csv
_deseq2_results = {}

if pdata is not None and _R_AVAILABLE:
    for (_num, _den) in DISEASE_CONTRASTS:
        # 检查该对比在 metadata 中的样本数
        _meta = pdata.obs
        _n_num = (_meta[DISEASE_COL] == _num).sum()
        _n_den = (_meta[DISEASE_COL] == _den).sum()

        if _n_num < 2 or _n_den < 2:
            print(f"  跳过 {_num} vs {_den}: 样本数不足 "
                  f"({_num}={_n_num}, {_den}={_n_den}, 最少需要各 2)")
            continue

        _cmd = [
            _RSCRIPT_PATH, "--vanilla", DESEQ2_RSCRIPT,
            os.path.abspath(DESEQ2_WORKDIR),
            DISEASE_COL, _num, _den,
        ]
        print(f"  执行: {' '.join(_cmd)}")
        try:
            _result = subprocess.run(
                _cmd, capture_output=True, text=True, timeout=600,
            )
            print(_result.stdout)
            if _result.returncode != 0:
                print(f"  Rscript 失败 (exit={_result.returncode}):")
                print(_result.stderr)
                continue

            _out_csv = os.path.join(
                DESEQ2_WORKDIR, f"deg_{_num}_vs_{_den}.csv"
            )
            if os.path.exists(_out_csv):
                _res_df = pd.read_csv(_out_csv, index_col=0)
                _deseq2_results[(_num, _den)] = _res_df
                _sig = (_res_df["padj"] < 0.05).sum()
                print(f"  完成: {len(_res_df)} 基因, {_sig} 个显著 (padj<0.05)")
            else:
                print(f"  完成但输出文件不存在: {_out_csv}")
        except subprocess.TimeoutExpired:
            print(f"  超时: {_num} vs {_den}（超过 10 分钟）")
        except Exception as _e:
            print(f"  异常: {_e}")

    print(f"\nDESeq2 完成: {len(_deseq2_results)}/{len(DISEASE_CONTRASTS)} 组对比")
elif not _R_AVAILABLE:
    print("=" * 60)
    print("DESeq2 对比已跳过——Rscript 不可用。")
    print("请确保: 1) R 已安装  2) scrna-integration-r env 已装好 DESeq2")
    print("3) environment-r.yml 中的 R 依赖已 conda env create")
    print("安装后重新运行本 cell。")
    print("=" * 60)
else:
    print("pdata 为 None，跳过 DESeq2")

## 结果展示

读回 DESeq2 结果，展示显著差异表达基因（DEG）摘要与火山图。

**为什么看火山图？** 火山图同时展示两个关键维度：
- **横轴（log2 Fold Change）**：生物学效应大小——基因在两组间的表达差异倍数
- **纵轴（-log10 adjusted p-value）**：统计学置信度——差异的统计显著性
- 右上角和左上角的点 = 同时满足大效应 + 高显著性的基因，是最值得优先验证的候选

**阈值解释**：
- `padj < 0.05`（Benjamini-Hochberg FDR 校正）：控制多重检验下的假发现率
- `|log2FC| > 1`（即 2 倍变化）：常见的效应量筛选阈值（本 cell 用 padj 着色，
  PI 可按需在火山图上叠加 log2FC 阈值线）

**结果去向**：
- 每个对比的完整 DEG 表写入 `results/tables/08_pseudobulk_deg_{num}_vs_{den}.csv`
- DESeq2 结果摘要写入 `adata.uns["08_pseudobulk_deg_v1"]`（含各对比的显著基因数、
  上调/下调计数等）


In [ ]:
# DESeq2 结果摘要 + 导出 CSV。
import datetime as _dt

_pb_uns = {
    "method": "decoupler pseudobulk + DESeq2 (subprocess Rscript)",
    "sample_col": SAMPLE_COL,
    "cell_type_col": CELL_TYPE_COL,
    "disease_col": DISEASE_COL,
    "contrasts": [],
    "timestamp": _dt.datetime.now().isoformat(),
    "r_available": _R_AVAILABLE,
}

for (_num, _den), _res_df in _deseq2_results.items():
    _sig = _res_df[_res_df["padj"] < 0.05]
    _up = (_sig["log2FoldChange"] > 0).sum()
    _dn = (_sig["log2FoldChange"] < 0).sum()

    _csv = f"results/tables/08_pseudobulk_deg_{_num}_vs_{_den}.csv"
    _res_df.to_csv(_csv)
    print(f"{_num} vs {_den}: {len(_sig)} sig DEG (up={_up}, down={_dn}) → {_csv}")

    _pb_uns["contrasts"].append({
        "numerator": _num,
        "denominator": _den,
        "n_sig": int(len(_sig)),
        "n_up": int(_up),
        "n_down": int(_dn),
        "csv": _csv,
    })

    # Top 10 展示
    _top = _sig.nsmallest(10, "padj")
    if len(_top) > 0:
        print(f"  Top 5: {', '.join(_top.index[:5])}")

if _deseq2_results:
    # 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 命名一致）
    adata.uns["stage"] = "08_pseudobulk_deg"     # 本 stage 标识
    adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致
    adata.uns["upstream"] = [UPSTREAM_PATH]
    adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"


    adata.uns["08_pseudobulk_deg_v1"] = _pb_uns
    print("\nDESeq2 结果已写入 adata.uns['08_pseudobulk_deg_v1']")
else:
    print("\n无 DESeq2 结果")

In [ ]:
# 火山图——展示第一个有显著 DEG 的对比（如有）。
if _deseq2_results:
    _plt_done = False
    for (_num, _den), _res_df in _deseq2_results.items():
        _sig = _res_df[_res_df["padj"] < 0.05]
        if len(_sig) == 0:
            continue

        fig, ax = plt.subplots(1, 1, figsize=(7, 6))
        _res_df["-log10_padj"] = -np.log10(_res_df["padj"].clip(lower=1e-300))
        _res_df["significant"] = _res_df["padj"] < 0.05

        ax.scatter(
            _res_df.loc[~_res_df["significant"], "log2FoldChange"],
            _res_df.loc[~_res_df["significant"], "-log10_padj"],
            s=3, color="grey", alpha=0.3, rasterized=True,
        )
        ax.scatter(
            _res_df.loc[_res_df["significant"], "log2FoldChange"],
            _res_df.loc[_res_df["significant"], "-log10_padj"],
            s=6, color="red", alpha=0.7, rasterized=True,
        )

        # 标注 top 10 基因
        _top = _sig.nsmallest(10, "padj")
        for _gn, _row in _top.iterrows():
            ax.annotate(
                _gn, (_row["log2FoldChange"], _row["-log10_padj"]),
                fontsize=6, alpha=0.9,
            )

        ax.axhline(-np.log10(0.05), ls="--", color="gray", lw=0.8)
        ax.set_title(f"Pseudobulk 差异表达：{_num} vs {_den}")
        ax.set_xlabel("log2 Fold Change")
        ax.set_ylabel("-log10(p_adj)")

        _vp = f"results/figures/08_pseudobulk_volcano_{_num}_vs_{_den}.png"
        fig.savefig(_vp, dpi=200, bbox_inches="tight")
        plt.show()
        plt.close("all")
        print(f"火山图已保存: {_vp}")
        _plt_done = True
        break   # 只画第一个有结果的对比

    if not _plt_done:
        print("所有对比无显著 DEG，跳过火山图")
else:
    print("无 DESeq2 结果，跳过可视化")

In [ ]:
# 内存自检。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

In [ ]:
# 写出 checkpoint（即使 DESeq2 没跑完，聚合后的 pdata 也值得保留）。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存。
del adata
if pdata is not None:
    del pdata
gc.collect()
print("内存已释放")